# Bookey 2/3 — Fine-tune Indic Parler-TTS for native Tanglish *style*

Starts from `ai4bharat/indic-parler-tts` (Apache-2.0) and fine-tunes it on the dataset from
notebook 1. Voices come from a **text description**, not a reference recording, so the result
speaks *like* native Tanglish speakers without copying anyone. Bookey then fixes two descriptions
(male / female teacher) as its house voices.

* Kaggle: Accelerator **GPU T4 x2** (or P100), Internet **on**. No secrets: the base model and data are attached as Kaggle inputs.
* Memory: the 24-layer decoder (~0.6 B params) is trained, the Flan-T5 text encoder is frozen;
  fp16 + gradient checkpointing + batch 2 fits a 16 GB GPU. If you hit out-of-memory, run this same
  notebook on a 24 GB GPU (Lightning AI / Modal free credits) and double `BATCH`.
* Sessions are 12 h max: checkpoints are pushed every `SAVE_STEPS`, and re-running resumes.

In [ ]:
!pip -q install "parler_tts[train] @ git+https://github.com/huggingface/parler-tts.git" accelerate jiwer
!pip -q install -U "protobuf>=5.29,<7"   # parler pins an old protobuf that transformers cannot import
!git clone -q https://github.com/huggingface/parler-tts.git /kaggle/working/parler-tts 2>/dev/null || true
# Upstream indexes the optional metadata dataset list even when there is none.
# pip also installs a copy of `training`, and that is the one the script imports.
!find /kaggle/working/parler-tts /usr/local/lib -path '*training/data.py' -exec sed -i 's/metadata_dataset_names\[i\]/(metadata_dataset_names[i] if metadata_dataset_names else None)/' {} +
# transformers' DacModel takes no `bandwidth` and returns codes without the leading chunk axis.
from pathlib import Path as _P
_run = _P("/kaggle/working/parler-tts/training/run_parler_tts_training.py")
_src = _run.read_text()
for _old, _new in (
    ('if bandwidth is not None:',
     'if bandwidth is not None and "bandwidth" in inspect.signature(audio_decoder.encode).parameters:'),
    ('output["labels"] = labels.squeeze(0).transpose(1, 2)',
     'output["labels"] = (labels if labels.dim() == 4 else labels.unsqueeze(0)).squeeze(0).transpose(1, 2)'),
):
    assert _old in _src, _old
    _src = _src.replace(_old, _new)
_run.write_text(_src)

In [ ]:
import glob, os, shutil
parts = sorted(glob.glob("/kaggle/input/**/part-*.parquet", recursive=True))
assert parts, "attach the output of bookey-1-tanglish-dataset"
BASE = os.path.dirname(glob.glob("/kaggle/input/**/model.safetensors", recursive=True)[0])
DATA_DIR = "/kaggle/tmp/tanglish"   # train-*/test-* parquet files: load_dataset() finds the splits by name
os.makedirs(DATA_DIR, exist_ok=True)
from datasets import load_dataset
raw = load_dataset("parquet", data_files=parts, split="train")
for name in ("train", "test"):
    raw.filter(lambda s, n=name: s == n, input_columns="split").remove_columns("split").to_parquet(f"{DATA_DIR}/{name}-00000-of-00001.parquet")
BATCH, ACCUM = 2, 16                              # effective batch 32
LR = 8e-5                                          # gentle: we adapt style, not relearn Tamil
EPOCHS = 3
SAVE_STEPS = 500
MAX_STEPS = 2500     # about 6-7 h at 9 s/step: fits one 12 h Kaggle session with audio encoding
SMOKE_TEST = False   # True: 60 steps to check memory + audio quality

In [ ]:
import subprocess, shlex
args = {
    "model_name_or_path": BASE,
    "feature_extractor_name": BASE,
    "description_tokenizer_name": "google/flan-t5-large",
    "prompt_tokenizer_name": BASE,
    "train_dataset_name": DATA_DIR, "train_dataset_config_name": "default", "train_split_name": "train",
    "eval_dataset_name": DATA_DIR, "eval_dataset_config_name": "default", "eval_split_name": "test",
    "target_audio_column_name": "audio", "description_column_name": "description",
    "prompt_column_name": "text",
    "max_duration_in_seconds": 15, "min_duration_in_seconds": 2, "max_text_length": 400,
    "do_train": True, "do_eval": True, "predict_with_generate": False,
    "per_device_train_batch_size": BATCH, "per_device_eval_batch_size": BATCH,
    "gradient_accumulation_steps": ACCUM, "gradient_checkpointing": True,
    "learning_rate": LR, "lr_scheduler_type": "cosine", "warmup_steps": 100,
    "adam_beta1": 0.9, "adam_beta2": 0.99, "weight_decay": 0.01,
    "num_train_epochs": EPOCHS, "freeze_text_encoder": True,
    "dtype": "float16", "audio_encoder_per_device_batch_size": 8,
    "eval_steps": SAVE_STEPS, "save_steps": SAVE_STEPS, "save_total_limit": 1, "logging_steps": 10, "max_steps": MAX_STEPS,
    "output_dir": "/kaggle/working/parler_out", "overwrite_output_dir": False,
    "temporary_save_to_disk": "/kaggle/tmp/audio_codes", "save_to_disk": "/kaggle/tmp/encoded",
    "preprocessing_num_workers": 4, "dataloader_num_workers": 4, "group_by_length": True,
    "push_to_hub": False,
    "report_to": "none", "seed": 42, "attn_implementation": "sdpa",
}
if SMOKE_TEST:
    args.update(max_steps=60, max_train_samples=400, max_eval_samples=32, eval_steps=60, save_steps=60,
                )
cli = " ".join(f"--{k} {shlex.quote(str(v).lower() if isinstance(v, bool) else str(v))}" for k, v in args.items())
cmd = f"cd /kaggle/working/parler-tts && accelerate launch --mixed_precision fp16 training/run_parler_tts_training.py {cli}"
print(cmd[:400], "...")
!{cmd}
assert os.path.isdir("/kaggle/working/parler_out"), "training failed: see the log above"

In [ ]:
# Listen: Bookey's two house voices, same sentence, base model vs fine-tuned.
import torch, soundfile as sf
from IPython.display import Audio, display
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

TEXT = "இன்னைக்கு நம்ம பைதான் வேரியபிள்ஸ் பத்தி கத்துக்கப்போறோம். ரொம்ப சிம்பிள் தான், கவலைப்படாதீங்க!"
VOICES = {
    "male": "A male teacher talks in a warm, conversational Tamil style, mixing English words naturally "
            "like a native Tanglish speaker, at a moderate pace. The recording is clear and close to the microphone.",
    "female": "A female teacher talks in a warm, conversational Tamil style, mixing English words naturally "
              "like a native Tanglish speaker, at a moderate pace. The recording is clear and close to the microphone.",
}
device = "cuda" if torch.cuda.is_available() else "cpu"
prompt_tok = AutoTokenizer.from_pretrained(BASE)
desc_tok = AutoTokenizer.from_pretrained("google/flan-t5-large")
for name, ckpt in [("base", BASE), ("tuned", "/kaggle/working/parler_out")]:
    try:
        model = ParlerTTSForConditionalGeneration.from_pretrained(ckpt).to(device)
    except Exception as e:
        print(name, "not available:", e); continue
    for gender, desc in VOICES.items():
        torch.manual_seed(0)
        d = desc_tok(desc, return_tensors="pt").to(device)
        p = prompt_tok(TEXT, return_tensors="pt").to(device)
        audio = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                               prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)
        wav = audio.cpu().numpy().squeeze()
        sf.write(f"/kaggle/working/{name}_{gender}.wav", wav, model.config.sampling_rate)
        print(name, gender); display(Audio(wav, rate=model.config.sampling_rate))
    del model; torch.cuda.empty_cache()

Download `tuned_male.wav` / `tuned_female.wav` (Output panel) and drop them into Bookey's voice test
page. If tuned beats base and today's engines, set `SMOKE_TEST = False`, run again (≈ 10–20 GPU-hours over a
few sessions), and Bookey will load `<you>/bookey-parler-tanglish` through its `indic_parler` provider.